# NCERT parsed-page EDA and preprocessing
Attach the private inventory and GPU parse notebook outputs. This notebook checks page identity and parsing coverage, then creates source-linked candidate passages. Review flagged pages before promoting content into the tutor.


In [1]:
import hashlib, json, re, statistics
from collections import Counter, defaultdict
from pathlib import Path

input_root = Path('/kaggle/input')
manifests = list(input_root.rglob('ncert_inventory/manifest.json'))
summaries = list(input_root.rglob('ncert_parsed/summary.json'))
assert len(manifests) == len(summaries) == 1, 'Attach one inventory and one successful full parse output'
manifest = json.loads(manifests[0].read_text())
summary = json.loads(summaries[0].read_text())
manifest = [row for row in manifest if row['file'] in summary['source_hashes']]
assert summary['chapters'] == len(manifest) and manifest
assert summary['pages'] == sum(row['pages'] for row in manifest)
assert summary['source_hashes'] == {row['file']: row['sha256'] for row in manifest}
assert not summary['errors']
parsed_root = summaries[0].parent
output = Path('/kaggle/working/ncert_prepared')
output.mkdir(exist_ok=True)


In [2]:
pages = []
for row in manifest:
    chapter = parsed_root / Path(row['file']).stem
    for number in range(1, row['pages'] + 1):
        page_dir = chapter / f'{number:04d}'
        status = (chapter / f'{number:04d}.done').read_text().strip()
        markdown_path = page_dir / 'markdown.md'
        assert status in {'parsed', 'empty'} and markdown_path.exists(), (row['file'], number)
        markdown = markdown_path.read_text(encoding='utf-8')
        assert (not markdown.strip()) == (status == 'empty'), (row['file'], number)
        flags = []
        if status == 'empty': flags.append('empty')
        if re.search(r'!\[[^]]*\]\([^)]*\)', markdown): flags.append('figure_requires_review')
        if '$' in markdown or re.search(r'\\(?:frac|sqrt|begin)', markdown): flags.append('math_requires_review')
        if '|' in markdown and re.search(r'^\s*\|', markdown, re.M): flags.append('table_requires_review')
        if '�' in markdown: flags.append('replacement_character')
        pages.append({**{k: row[k] for k in ('grade', 'subject', 'book_code', 'chapter', 'file', 'sha256', 'url')},
                      'page': number, 'status': status, 'characters': len(markdown), 'flags': flags,
                      'markdown_path': str(markdown_path.relative_to(parsed_root))})
assert len(pages) == summary['pages']
assert len({(p['file'], p['page']) for p in pages}) == len(pages)
with (output / 'page_audit.jsonl').open('w', encoding='utf-8') as stream:
    for page in pages: stream.write(json.dumps(page) + '\n')
counts = Counter(flag for page in pages for flag in page['flags'])
report = {'source_chapters': len(manifest), 'physical_pages': len(pages), 'nonempty_pages': sum(p['status'] == 'parsed' for p in pages),
          'empty_pages': sum(p['status'] == 'empty' for p in pages), 'flag_counts': counts,
          'by_grade_subject': {f'{g}_{s}': {'chapters': len([r for r in manifest if (r['grade'], r['subject']) == (g, s)]),
                                            'pages': len([p for p in pages if (p['grade'], p['subject']) == (g, s)])}
                               for g, s in sorted({(r['grade'], r['subject']) for r in manifest})}}
(output / 'eda_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


{
  "source_chapters": 130,
  "physical_pages": 3066,
  "nonempty_pages": 3047,
  "empty_pages": 19,
  "flag_counts": {
    "figure_requires_review": 2303,
    "math_requires_review": 1324,
    "table_requires_review": 385,
    "replacement_character": 7,
    "empty": 19
  },
  "by_grade_subject": {
    "6_maths": {
      "chapters": 10,
      "pages": 392
    },
    "6_science": {
      "chapters": 12,
      "pages": 254
    },
    "7_maths": {
      "chapters": 15,
      "pages": 511
    },
    "7_science": {
      "chapters": 12,
      "pages": 188
    },
    "8_maths": {
      "chapters": 14,
      "pages": 423
    },
    "8_science": {
      "chapters": 13,
      "pages": 228
    },
    "9_maths": {
      "chapters": 14,
      "pages": 364
    },
    "9_science": {
      "chapters": 13,
      "pages": 272
    },
    "10_maths": {
      "chapters": 14,
      "pages": 217
    },
    "10_science": {
      "chapters": 13,
      "pages": 217
    }
  }
}


In [3]:
# Split on Markdown headings, keeping the exact source page and original text.
def usable_heading(value):
    # shortcut: OCR headings are noisy, upgrade after reviewing full-corpus heading samples.
    value = value.strip()
    return (bool(re.search(r'[A-Za-z]{3}', value)) and len(value) <= 100 and
            not re.search(r'[?=_]', value) and
            not re.match(r'(?i)^(?:figure it out|page no\.?\s*\d+|teacher.s note|summary|rules|explanation|more examples|or|section\s+\d+(?:\.\d+)?|q\.?\s*\d+|example\s*\d+|exercise\s*\d*)\b', value))

assert usable_heading('3.6 Locating and Comparing Decimals')
assert not any(usable_heading(x) for x in ('Figure it Out', 'Page No. 50', 'Q.4 Solve these:', 'What is its length?'))
passages = []
last_heading = {}
for page in pages:
    if page['status'] == 'empty': continue
    markdown = (parsed_root / page['markdown_path']).read_text(encoding='utf-8')
    sections = re.split(r'(?=^#{1,4}\s+)', markdown, flags=re.M)
    for section_index, section in enumerate(sections):
        section = section.strip()
        if not section: continue
        raw_heading = next((line.lstrip('# ').strip() for line in section.splitlines() if line.startswith('#')), '')
        if usable_heading(raw_heading): last_heading[page['file']] = raw_heading
        heading = last_heading.get(page['file'], f"Class {page['grade']} {page['subject'].title()} Chapter {page['chapter']}")
        paragraphs = re.split(r'\n\s*\n', section)
        batch = ''
        for paragraph in paragraphs + ['']:
            if batch and (not paragraph or len(batch) + len(paragraph) > 1600):
                passage_id = hashlib.sha256(f"{page['sha256']}:{page['page']}:{section_index}:{len(passages)}:{batch}".encode()).hexdigest()[:24]
                flags = page['flags'] + (['answer_risk'] if re.search(r'(?im)^\s*(?:#{1,4}\s*)?(?:answers?|solutions?)\b', batch) else [])
                passages.append({'passage_id': passage_id, 'grade': page['grade'], 'subject': page['subject'],
                                 'book_code': page['book_code'], 'chapter': page['chapter'], 'page': page['page'],
                                 'source_file': page['file'], 'source_sha256': page['sha256'], 'source_url': page['url'],
                                 'heading': heading, 'text': batch.strip(), 'asset_dir': str(Path(page['markdown_path']).parent),
                                 'review_flags': flags, 'review_status': 'candidate'})
                batch = ''
            batch += ('\n\n' if batch else '') + paragraph
with (output / 'candidate_passages.jsonl').open('w', encoding='utf-8') as stream:
    for passage in passages: stream.write(json.dumps(passage, ensure_ascii=False) + '\n')
assert len({p['passage_id'] for p in passages}) == len(passages)
print('Candidate passages:', len(passages), 'Median characters:', statistics.median(len(p['text']) for p in passages))


Candidate passages: 7221 Median characters: 584
